<a href="https://colab.research.google.com/github/rajukarki467/Data-Science-projects/blob/main/FakeNewsDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # **Fake News Detection — Data Science Project**

*MSc in Informatics and Intelligent Systems Engineering*

Institute: Thapathali Campus, IOE, Tribhuvan University

Course: Data Science and Modeling



Student: Raju Karki
Roll No: MSISE012

# Import Libraries
* IMPORT ALL REQUIRED LIBRARIES

In [3]:
# Core data libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
#standard libraries
import re # for regural expression
import string
import warnings
warnings.filterwarnings('ignore')

NLP - natural language Processing

In [5]:
import nltk
nltk.download('stopwords',quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [6]:
# Feature Engineering
from sklearn.feature_extraction.text import(
    TfidfTransformer,
    CountVectorizer,
    TfidfVectorizer
)

In [7]:
# machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

In [8]:
# Model Selection and Cross-Validation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

In [9]:
# Evaluation Metrics
import sklearn.metrics as metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)


In [10]:
# model persistence
import joblib

In [11]:
print("All libraries imported sucessfully.")
print(f"Numpy :{np.__version__}")
print(f"Pandas: {pd.__version__}")
import sklearn
print(f' Sklearn : {sklearn.__version__}')

All libraries imported sucessfully.
Numpy :2.0.2
Pandas: 2.2.2
 Sklearn : 1.6.1


#Load Dataset

 **Files required:** `True.csv` and `Fake.csv` (place in same directory as this notebook)

| File | Content | Source |
|---|---|---|
| `True.csv` | Real news articles | Reuters, politicsNews, worldnews |
| `Fake.csv` | Fake news articles | Conspiracy, Government News, tabloids |

# Read the dataset using read_csv-
*  engine='python'       : handles complex CSV encoding
*  quotechar='"'         : standard double-quote delimiter
*  doublequote=True      : handles escaped quotes inside fields
*  on_bad_lines='skip'   : skip malformed rows gracefully

In [12]:
true_news = pd.read_csv(
    '/content/True.csv',
    engine='python',
    quotechar='"',
    doublequote=True,
    on_bad_lines='skip'
)

fake_news = pd.read_csv(
    '/content/Fake.csv',
    engine='python',
    quotechar='"',
    doublequote=True,
    on_bad_lines='skip'
)

 # Exploratory Data Analysis (EDA):
* Exploring both datasets to understand structure, statistics, and distributions before cleaning.

In [13]:
print(f"Datasets loaded sucessfully.")
print(f"true news shape  :  {true_news.shape}")
print(f"Fake news shape  :  {fake_news.shape}")
print(f"Columns          :  {true_news.columns.tolist()}")
print(f"Columns          :  {fake_news.columns.tolist()}")

Datasets loaded sucessfully.
true news shape  :  (21417, 4)
Fake news shape  :  (23481, 4)
Columns          :  ['title', 'text', 'subject', 'date']
Columns          :  ['title', 'text', 'subject', 'date']


# 1. EXPLORE TRUE NEWS DATASET

In [14]:
# Show top 5 rows of the data for basic information
print("Top 5 rows of True News:")
true_news.head()

Top 5 rows of True News:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [15]:
# Knowing the dataset shape
print(" True News shape (rows, columns):")
print(true_news.shape)

 True News shape (rows, columns):
(21417, 4)


In [16]:
# Show the last 5 rows
print("Last 5 rows of True News:")
true_news.tail()

Last 5 rows of True News:


,title,text,subject,date
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017"
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017"
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017"
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017"
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017"


* Basic statistics: count, unique, top, freq for categorical;
* count, mean, std, min, max for numerical

In [17]:
print("True News — Basic Statistics:")
true_news.describe(include='all')

True News — Basic Statistics:


,title,text,subject,date
count,21417,21417,21417,21417
unique,20826,21192,2,716
top,Factbox: Trump fills top jobs for his administ...,(Reuters) - Highlights for U.S. President Dona...,politicsNews,"December 20, 2017"
freq,14,8,11272,182


Checking data basic info:
* gives detail of memory consumption, datatypes,
*  total number of rows, column names, and null value count

In [18]:

print(" True News — Info (dtypes, memory, null counts):")
true_news.info()

 True News — Info (dtypes, memory, null counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [19]:
# Showing total count of datasets (non-null values per column)
print("rue News — Column-wise non-null count:")
print(true_news.count())

rue News — Column-wise non-null count:
title      21417
text       21417
subject    21417
date       21417
dtype: int64


# 2.  EXPLORE FAKE NEWS DATASET

In [20]:
print(" Top 5 rows of Fake News:")
fake_news.head()

 Top 5 rows of Fake News:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [22]:
# Show the last 5 rows
print("Last 5 rows of Fake News:")
fake_news.tail()

Last 5 rows of Fake News:


,title,text,subject,date
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016"


In [23]:
# Shape of fake news dataset
print("Fake News shape (rows, columns):")
print(fake_news.shape)

Fake News shape (rows, columns):
(23481, 4)


In [24]:
# Basic statistics of fake news data
print("Fake News — Basic Statistics:")
fake_news.describe(include='all')

Fake News — Basic Statistics:


,title,text,subject,date
count,23481,23481,23481,23481
unique,17903,17455,6,1681
top,MEDIA IGNORES Time That Bill Clinton FIRED His...,,News,"May 10, 2017"
freq,6,626,9050,46


In [25]:
# Data info of fake news
print("Fake News — Info (dtypes, memory, null counts):")
fake_news.info()

Fake News — Info (dtypes, memory, null counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [26]:
# Count of fake news columns
print("Fake News — Column-wise non-null count:")
print(fake_news.count())

Fake News — Column-wise non-null count:
title      23481
text       23481
subject    23481
date       23481
dtype: int64


# CLASS BALANCE OBSERVATION

In [29]:
print(f"True news records  : {len(true_news):,}")
print(f"Fake news records  : {len(fake_news):,}")
print()

True news records  : 21,417
Fake news records  : 23,481



# Observation:
  * True = 21,417  
  *   Fake = 23,500
  # Analysis:
  * Dataset is near-balanced — no class imbalance handling needed.
  (SMOTE, class_weight adjustment are NOT required.)
  * We can able to evaluate a model based on the accuracy  of the model